<a href="https://colab.research.google.com/github/kjmobile/lb2/blob/main/2_LM_Multiple_Regression_one_hot_encoding_Airfares_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Multiple Regression with a Categorical Variable: Airfares

The airline that leads a market is a **categorical variable**: its values are names (AA, DL, UA, ...), not
numbers. This notebook puts it into a multiple regression in the two ways shown in the W3.1 slides,
**one-hot encoding** and **dummy variables**, and uses each one where it belongs.

- **Predict (y):** `avg_fare`, the average fare of a market
- **From (X):** `distance_mi`, `lcc_share`, `hhi` (numbers) and `top_airline` (text: the airline with the most
  passengers)

| | Part 1. Inference | Part 2. Prediction |
|---|---|---|
| Question | Do fares differ by the leading airline, with distance, low-cost share and concentration held fixed? | Does knowing the leading airline help predict the fares of markets the model has not seen? |
| Evidence | coefficients and p-values, all 200 markets | the error on 40 test markets |
| Airline enters as | dummy variables | one-hot encoding |
| Tool | `statsmodels` | `scikit-learn` |

These are the same two approaches as in notebook 0.

> **Data.** The 200 busiest U.S. domestic markets in 2019, read straight from the course GitHub repository.
> Source: Sabre Market Intelligence, licensed to Embry-Riddle.

> **Figures.** The figures in this notebook are pictures made by its full version (`2_LM_Multiple_Regression_one_hot_encoding_Airfares_all.ipynb`), so you can focus on the code that fits the model and produces the results.

## 1. Data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

routes = pd.read_csv('https://raw.githubusercontent.com/kjmobile/lb2/main/data/sabre_routes_2019.csv')
routes.head()

,airport_1,airport_2,distance_mi,passengers,avg_fare,lcc_share,hhi,top_airline
0,JFK,LAX,2470.0,3115042.0,349.29,0.321,0.269,DL
1,LGA,ORD,731.0,2600397.0,174.47,0.086,0.293,AA
2,LAX,SFO,337.0,2289910.0,104.55,0.203,0.212,AS
3,LAX,ORD,1741.0,1853120.0,187.84,0.146,0.332,UA
4,LAS,LAX,236.0,1831441.0,73.20,0.438,0.180,WN


| Column | Meaning |
|---|---|
| `distance_mi` | distance in miles |
| `lcc_share` | share of passengers on low-cost airlines |
| `hhi` | market concentration (1 = one airline carries everyone) |
| `top_airline` | the airline with the most passengers: AA American, AS Alaska, B6 JetBlue, DL Delta, HA Hawaiian, NK Spirit, UA United, WN Southwest |
| `avg_fare` | average fare ($), what we predict |

**What is `hhi`?** The **Herfindahl–Hirschman Index**, the standard measure of market concentration.
Square each airline's share of the market's passengers, then add them up:

$$HHI = \sum_i s_i^2 \qquad (s_i = \text{airline } i\text{'s share of passengers})$$

| Market | Shares | HHI |
|---|---|---|
| One airline carries everyone | 1.0 | 1.0 |
| Two airlines, half each | 0.5, 0.5 | 0.25 + 0.25 = 0.5 |
| Four airlines, a quarter each | 0.25 each | 4 × 0.0625 = 0.25 |

Here shares are fractions, so HHI runs from 0 to 1. News reports and U.S. merger reviews put shares in percent,
so the same index runs from 0 to 10,000 (two airlines at half each = 5,000).

In [2]:
data = routes[['distance_mi', 'lcc_share', 'hhi', 'top_airline', 'avg_fare']]
data['top_airline'].value_counts()

top_airline
WN    45
DL    44
AA    39
UA    35
AS    19
B6    10
HA     5
NK     3
Name: count, dtype: int64

Average fare, distance and low-cost share by leading airline:

In [3]:
data.groupby('top_airline')[['avg_fare', 'distance_mi', 'lcc_share']].mean().round(2)

,avg_fare,distance_mi,lcc_share
top_airline,,,
AA,170.98,1195.97,0.21
AS,154.69,1038.89,0.17
B6,166.57,1333.90,0.72
DL,187.95,1173.68,0.30
HA,156.23,1080.00,0.08
NK,155.55,1430.33,0.60
UA,222.89,1403.31,0.24
WN,117.82,630.62,0.74


United-led markets average about \$223 and Southwest-led markets about \$118, but United's markets are also more
than twice as long (1,403 vs. 631 miles) and have far fewer low-cost passengers (0.24 vs. 0.74). Comparing these
averages directly is not fair. A multiple regression compares the airlines with distance, low-cost share and
concentration **held fixed**.

## 2. One-hot Encoding and Dummy Variables

`top_airline` is text, and a regression needs numbers. Numbering the airlines 1 to 8 would invent an order
(is UA "more" than DL?). Instead, each airline becomes a 0/1 column, in one of two ways:

| | One-hot encoding | Dummy variables |
|---|---|---|
| Airline columns | one for every airline: 8 | one airline dropped: 7 |
| The dropped airline | none | becomes the **reference** |
| Multicollinearity (with an intercept) | yes | no |

In [4]:
# One-hot encoding: a 0/1 column for every airline (8 columns)
data_onehot = pd.get_dummies(data, columns=['top_airline'], dtype=int)
data_onehot.head(3)

,distance_mi,lcc_share,hhi,avg_fare,top_airline_AA,top_airline_AS,top_airline_B6,top_airline_DL,top_airline_HA,top_airline_NK,top_airline_UA,top_airline_WN
0,2470.0,0.321,0.269,349.29,0,0,0,1,0,0,0,0
1,731.0,0.086,0.293,174.47,1,0,0,0,0,0,0,0
2,337.0,0.203,0.212,104.55,0,1,0,0,0,0,0,0


In [5]:
# Dummy variables: drop_first=True drops the first airline (AA), which becomes the reference (7 columns)
data_dummy = pd.get_dummies(data, columns=['top_airline'], dtype=int, drop_first=True)
data_dummy.head(3)

,distance_mi,lcc_share,hhi,avg_fare,top_airline_AS,top_airline_B6,top_airline_DL,top_airline_HA,top_airline_NK,top_airline_UA,top_airline_WN
0,2470.0,0.321,0.269,349.29,0,0,1,0,0,0,0
1,731.0,0.086,0.293,174.47,0,0,0,0,0,0,0
2,337.0,0.203,0.212,104.55,1,0,0,0,0,0,0


Why does one-hot encoding cause multicollinearity? Add up the eight airline columns in each row:

In [6]:
data_onehot.filter(like='top_airline_').sum(axis=1).value_counts()

1    200
Name: count, dtype: int64

All 200 rows add up to **1**, because each market has exactly one leading airline. The intercept is also a 1 in
every row, so the eight columns and the intercept carry the same information twice. The model can still
predict, but it cannot tell how much of the base fare belongs to the intercept and how much to each airline,
so the airline coefficients cannot be read. Dropping one column (dummy variables) removes the overlap.

## Part 1. Inference: Do Fares Differ by the Leading Airline?

Inference reads coefficients and their p-values, so it needs **dummy variables** and uses all 200 markets.
In a statsmodels formula, `C(top_airline)` makes the dummy variables for you. The first airline in alphabetical
order, **AA (American)**, becomes the reference, so it has no term of its own:

$$\widehat{\text{fare}} = a + b_1 \times \text{distance} + b_2 \times \text{lcc\_share} + b_3 \times \text{hhi}
+ c_{AS} \times \text{AS} + c_{B6} \times \text{B6} + \dots + c_{WN} \times \text{WN}$$

In [7]:
results = smf.ols('avg_fare ~ distance_mi + lcc_share + hhi + C(top_airline)', data=data).fit()
print(results.summary().tables[1])   # the coefficient table of the summary

                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               77.8669      9.560      8.145      0.000      59.008      96.726
C(top_airline)[T.AS]    -2.6048      8.122     -0.321      0.749     -18.626      13.416
C(top_airline)[T.B6]    29.3194     11.875      2.469      0.014       5.896      52.743
C(top_airline)[T.DL]    31.1660      6.449      4.833      0.000      18.445      43.887
C(top_airline)[T.HA]   -24.1927     13.693     -1.767      0.079     -51.204       2.819
C(top_airline)[T.NK]    10.8555     17.863      0.608      0.544     -24.382      46.093
C(top_airline)[T.UA]    40.1526      6.679      6.012      0.000      26.979      53.327
C(top_airline)[T.WN]    26.9666      8.852      3.046      0.003       9.504      44.429
distance_mi              0.0692      0.004     19.692      0.000       0.062       0.076
lcc_share            

**Read it.** Each airline coefficient is a difference from an **American-led** market with the same distance,
low-cost share and concentration.

- `C(top_airline)[T.UA]` is about 40.2 with p < 0.001: a United-led market averages about \$40 more.
- `C(top_airline)[T.AS]` is about -2.6 with p = 0.749: no clear evidence that Alaska-led markets differ from
  American-led ones.

These are differences the model finds, not their causes. Anything the model leaves out, such as the share of
business travelers, ends up in these coefficients.

### Your turn

Make **Delta (DL)** the reference airline instead of American, and compare United with Delta.

1. In the cell below, fill in the blank (`___`) and run it.
2. Then finish this sentence:
   *With Delta as the reference, the American coefficient is ___ and the United coefficient is ___ (p-value ___).*

<details><summary>Show answer</summary>

```python
results_dl = smf.ols('avg_fare ~ distance_mi + lcc_share + hhi + C(top_airline, Treatment(reference="DL"))', data=data).fit()
print(results_dl.summary().tables[1])
```

*With Delta as the reference, the American coefficient is **-31.1660** (about -31.2) and the United coefficient is **8.9866** (about 9.0), with a p-value of **0.176**. A United-led market averages about 9 dollars more than a Delta-led market with the same distance, low-cost share and concentration, but the p-value gives no clear evidence of a difference. The distance, low-cost share and concentration coefficients do not change.*
</details>

In [ ]:
results_dl = smf.ols('avg_fare ~ distance_mi + lcc_share + hhi + C(top_airline, Treatment(reference="___"))', data=data).fit()
print(results_dl.summary().tables[1])

## Part 2. Prediction: Does the Airline Help Predict Fares?

Prediction is judged by the error on markets the model has not seen, so 20% of the markets are held out as a
test set, as in notebook 0. Here the airline enters as **one-hot encoding**: the coefficients are not read, so no
reference airline is needed.

$$\widehat{\text{fare}} = a + b_1 \times \text{distance} + b_2 \times \text{lcc\_share} + b_3 \times \text{hhi}
+ c_{AA} \times \text{AA} + c_{AS} \times \text{AS} + \dots + c_{WN} \times \text{WN}$$

In [9]:
# Inputs: every column except the fare (3 numbers + 8 one-hot airline columns)
X = data_onehot.drop(columns='avg_fare')
y = data_onehot['avg_fare']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print('train:', len(X_train), ' test:', len(X_test))

train: 160  test: 40


In [10]:
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

mse = mean_squared_error(y_test, pred)
print('RMSE:', round(mse ** 0.5, 1), 'dollars')
print('test R2:', round(model.score(X_test, y_test), 3))

RMSE: 30.9 dollars
test R2: 0.744


**Figure 1**

![Figure 1](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/2_fig1.png)

Points near the red line are good predictions. A typical prediction misses by about the RMSE.

### Did the airline help?

In [11]:
numbers = ['distance_mi', 'lcc_share', 'hhi']
model_num = LinearRegression()
model_num.fit(X_train[numbers], y_train)

print('test R2, numbers only:    ', round(model_num.score(X_test[numbers], y_test), 3))
print('test R2, plus the airline:', round(model.score(X_test, y_test), 3))

test R2, numbers only:     0.697
test R2, plus the airline: 0.744


Adding the leading airline raises the test R² from 0.697 to 0.744: knowing the airline improves predictions on
markets the model has not seen.

### One-hot encoding or dummy variables?

In [12]:
X_dummy = data_dummy.drop(columns='avg_fare')
X_dummy_train, X_dummy_test = train_test_split(X_dummy, test_size=0.2, random_state=0)   # same split as above

model_dummy = LinearRegression()
model_dummy.fit(X_dummy_train, y_train)

print('test R2, one-hot encoding:', round(model.score(X_test, y_test), 3))
print('test R2, dummy variables: ', round(model_dummy.score(X_dummy_test, y_test), 3))

test R2, one-hot encoding: 0.744
test R2, dummy variables:  0.744


Exactly the same. The dropped column adds no information: a market with 0 in all seven dummy columns is an
American-led market. For **prediction** the two encodings are interchangeable; for **inference** they are not,
because only dummy variables give airline coefficients you can read.

## Summary

| | Part 1. Inference | Part 2. Prediction |
|---|---|---|
| Airline enters as | dummy variables (reference: AA) | one-hot encoding |
| Answer | United-led markets average about \$40 more than American-led ones (p < 0.001) | the airline raises test R² from 0.697 to 0.744 |

| | pandas (columns for scikit-learn) | statsmodels formula |
|---|---|---|
| One-hot encoding | `pd.get_dummies(data, columns=['top_airline'])` | `C(top_airline) - 1` (no intercept) |
| Dummy variables | `pd.get_dummies(data, columns=['top_airline'], drop_first=True)` | `C(top_airline)` |
| Change the reference | drop that airline's column instead | `C(top_airline, Treatment(reference="DL"))` |

---
# Challenge

> In Part 2, `model` has all eight one-hot columns **and** an intercept. Fit the same columns with no intercept:
>
> ```python
> model_1 = LinearRegression(fit_intercept=False)
> model_1.fit(X_train, y_train)
> ```
>
> Compare `model_1.coef_` with `model.coef_`, and `model_1.score(X_test, y_test)` with `model.score(X_test, y_test)`.

- Which coefficients change, and which stay the same? Do the predictions change?
- What does each airline coefficient of `model_1` mean? (Hint: section 2. Without an intercept, nothing overlaps
  with the eight airline columns.)

In [13]:
# Your answer